# 두 운동 정지중력으로 yaw 포함 R 직접 결정 (R_kabsch2)

중력 1개는 yaw(중력축 둘레 회전)를 못 잡지만, **자세가 다른 두 운동**(deadlift, latpulldown)의
정지중력 2개가 비공선이면 paired Kabsch 로 3DOF(yaw 포함)가 닫힌형으로 결정된다.

규칙(단순·일관):
- 운동 2개: deadlift, latpulldown 고정
- 사람 2명: source 1명 + target 1명, 각자 두 운동 모두 담당
- 정지시각: 비디오 주석 없이 **데이터기반 자동검출**(세션별 최저분산 구간, `REGION='head'`=시작 휴식부)
- 한 사람이 한 운동에 여러 세션이면 **본인 세션들만** 평균 (다른 사람 안 섞음)

확정 캘리(민감도로 결정): **같은 사람 sub7→7** (잔차 10.6°, 다른 두 사람보다 깨끗).
결과 R(target→source, `imu @ R.T`)을 `results/R_matrices/R_kabsch2.npy` 로 저장 →
`python data_preprocess_MM.py --method kabsch2 --R results/R_matrices/R_kabsch2.npy`.

> 동치 스크립트: `scripts/build_R_kabsch2_twosubject.py` (CLI 버전).


In [ ]:
import torch
# Apple MPS 환경 노트북. R 계산 자체는 모델/디바이스가 필요 없지만(순수 numpy),
# 규약 일치 + 추후 다운스트림 평가 확장 대비로 device 만 잡아둔다.
DEV = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', DEV)

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = os.path.abspath('../..')
sys.path.insert(0, ROOT)
from eval_utils import IMU_COLS, rotation_align   # 읽기 전용

EXERCISES = ['deadlift', 'latpulldown']
print('IMU_COLS =', IMU_COLS)

In [ ]:
def unit(v):
    v = np.asarray(v, float)
    return v / (np.linalg.norm(v) + 1e-12)

def angle_deg(a, b):
    return float(np.degrees(np.arccos(np.clip(np.dot(unit(a), unit(b)), -1, 1))))

def kabsch(src, tgt):
    """src_i ~= R @ tgt_i 인 proper rotation R (det=+1). 행벡터: src ~= tgt @ R.T."""
    H = np.asarray(tgt).T @ np.asarray(src)
    Um, _, Vt = np.linalg.svd(H); V = Vt.T
    d = np.sign(np.linalg.det(V @ Um.T))
    return V @ np.diag([1, 1, d]) @ Um.T

def stillest_window(A, w):
    """A:(T,3) 에서 길이 w 최저분산(가장 정지) 구간의 (평균중력3, rel_std)."""
    T = len(A); w = min(w, T)
    if w < 3:
        m = A.mean(0); return m, float(np.linalg.norm(A.std(0)) / (np.linalg.norm(m) + 1e-12))
    cs  = np.cumsum(np.vstack([np.zeros(3), A]), 0)
    cs2 = np.cumsum(np.vstack([np.zeros(3), A * A]), 0)
    means  = (cs[w:]  - cs[:-w])  / w
    meansq = (cs2[w:] - cs2[:-w]) / w
    stdnorm = np.sqrt(np.clip(meansq - means ** 2, 0, None).sum(1))
    i = int(np.argmin(stdnorm)); m = means[i]
    return m, float(stdnorm[i] / (np.linalg.norm(m) + 1e-12))

In [ ]:
# 필요한 컬럼만 로드 (samsung1=source, samsung2=target)
s1 = pd.read_parquet(os.path.join(ROOT, 'data', 'samsung1.parquet'),
                     columns=IMU_COLS + ['filename', 'timestamp', 'subject', 'exercise'])
s2 = pd.read_parquet(os.path.join(ROOT, 'data', 'samsung2.parquet'),
                     columns=IMU_COLS + ['csv_filename_l', 'Index_Time', 'subject_id', 'exercise'])
print('s1', s1.shape, '| s2', s2.shape)

In [ ]:
# ── 파라미터 (확정 캘리: 같은 사람 sub7 -> 7, 시작부 정지검출) ──
SRC_SUBJECT = 'sub7'   # samsung1 source 피험자 1명
TGT_SUBJECT = 7        # samsung2 target 피험자 1명(id)
WIN_SEC     = 0.3      # 정지검출 윈도우 길이(초)
REL_THRESH  = 0.10     # 세션 채택 임계(최저분산구간 rel_std < 이 값)
REGION      = 'head'   # 'all'|'head'|'tail'  동적운동은 head(시작 휴식)로 국면 고정
REGION_FRAC = 0.3

def subject_exercise_gravity(df, sess_col, time_col, subject_col, subject, exercise):
    sub = df[(df[subject_col] == subject) & (df['exercise'] == exercise)]
    if not len(sub):
        raise ValueError(f'데이터 없음: subject={subject} exercise={exercise}')
    units, rels, used = [], [], 0
    for _, g in sub.groupby(sess_col):
        g = g.sort_values(time_col)
        t = g[time_col].to_numpy(float)
        dt = np.median(np.diff(t)) if len(t) > 1 else 1.0
        fs = 1.0 / dt if dt > 0 else 1.0
        w = max(5, int(round(WIN_SEC * fs)))
        A = g[IMU_COLS].to_numpy(np.float64); T = len(A)
        if REGION == 'head':
            A = A[:max(w, int(REGION_FRAC * T))]
        elif REGION == 'tail':
            A = A[min(T - w, int((1 - REGION_FRAC) * T)):]
        mean, rel = stillest_window(A, w)
        if rel <= REL_THRESH:
            units.append(unit(mean)); rels.append(rel); used += 1
    if not units:
        raise ValueError(f'정지구간 없음(rel>{REL_THRESH}): {subject}/{exercise}')
    units = np.vstack(units); g_mean = unit(units.mean(0))
    disp = float(np.mean([angle_deg(u, g_mean) for u in units]))
    print(f'  {exercise:12s} | 세션 {used}/{sub[sess_col].nunique()} 사용 | '
          f'평균중력 {np.round(g_mean,3).tolist()} | 세션간 산포 {disp:.1f}° | rel_std {np.mean(rels):.3f}')
    return g_mean

## 정지중력 자동검출 (sub7 → 7, region=head)

In [ ]:
print(f'[source samsung1] subject={SRC_SUBJECT}')
g1 = {ex: subject_exercise_gravity(s1, 'filename', 'timestamp', 'subject', SRC_SUBJECT, ex)
      for ex in EXERCISES}
print(f'[target samsung2] subject={TGT_SUBJECT}')
g2 = {ex: subject_exercise_gravity(s2, 'csv_filename_l', 'Index_Time', 'subject_id', TGT_SUBJECT, ex)
      for ex in EXERCISES}

## yaw 관측가능성 · rigid 정합 · paired Kabsch R

In [ ]:
# ── 관측가능성 + rigid 정합 + paired Kabsch ──
ang_s1 = angle_deg(g1['deadlift'], g1['latpulldown'])
ang_s2 = angle_deg(g2['deadlift'], g2['latpulldown'])
print(f'source angle(deadlift, latpulldown) = {ang_s1:.1f}°')
print(f'target angle(deadlift, latpulldown) = {ang_s2:.1f}°')
print(f'디바이스 간 사이각 불일치 |Δ| = {abs(ang_s1-ang_s2):.1f}°  (작을수록 순수 디바이스축 회전)')

P1 = np.vstack([g1['deadlift'], g1['latpulldown']])   # source
P2 = np.vstack([g2['deadlift'], g2['latpulldown']])   # target
R  = kabsch(P1, P2)                                     # target→source, imu @ R.T
res = [angle_deg(P1[i], P2[i] @ R.T) for i in range(2)]
sv  = np.linalg.svd(P2.T @ P1, compute_uv=False)
yaw_info = float(sv[1] / (sv[0] + 1e-12))

R_tilt = rotation_align(g2['latpulldown'], g1['latpulldown'])
tilt_res_dl = angle_deg(g1['deadlift'], g2['deadlift'] @ R_tilt.T)

print('\nR (target→source, imu @ R.T):'); print(np.round(R, 4))
print(f'det(R) = {np.linalg.det(R):+.4f}')
print(f'정렬 잔차: deadlift {res[0]:.2f}° · latpulldown {res[1]:.2f}°')
print(f'특이값 = {np.round(sv,3).tolist()} | yaw 정보량 sv2/sv1 = {yaw_info:.3f} (0이면 yaw 못잡음)')
print(f'[비교] 단일중력 tilt-only 면 deadlift 가 {tilt_res_dl:.1f}° 어긋남 → 그만큼이 yaw 로 잡혀야 할 양')

In [ ]:
# ── 시각화: 두 중력벡터 (source vs R로 정렬한 target) ──
fig = plt.figure(figsize=(11, 5))
axL = fig.add_subplot(121, projection='3d'); axR = fig.add_subplot(122, projection='3d')

def draw(ax, vecs, title):
    o = np.zeros(3)
    for (name, v, c) in vecs:
        ax.quiver(*o, *v, color=c, lw=2, arrow_length_ratio=0.12, label=name)
    ax.set_xlim(-1,1); ax.set_ylim(-1,1); ax.set_zlim(-1,1)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_title(title); ax.legend(fontsize=8)

draw(axL, [('s1 deadlift', g1['deadlift'], 'tab:blue'),
           ('s1 latpulldown', g1['latpulldown'], 'tab:green')],
     f'SOURCE 정지중력 (angle {ang_s1:.0f}°)')
draw(axR, [('s1 deadlift', g1['deadlift'], 'tab:blue'),
           ('s1 latpulldown', g1['latpulldown'], 'tab:green'),
           ('s2→R deadlift', g2['deadlift'] @ R.T, 'tab:orange'),
           ('s2→R latpulldown', g2['latpulldown'] @ R.T, 'tab:red')],
     f'R 정렬 후 겹침 (잔차 {res[0]:.1f}°)')
plt.tight_layout(); plt.show()

## R 저장

In [ ]:
# ── 저장: R_kabsch2.npy (규약: aligned = imu @ R.T, R=target→source) ──
out = os.path.join(ROOT, 'results', 'R_matrices', 'R_kabsch2.npy')
os.makedirs(os.path.dirname(out), exist_ok=True)
np.save(out, np.asarray(R, np.float64))
print('저장:', out)
print('다음:')
print(f'  python data_preprocess_MM.py --method kabsch2 --R {out}')
print('  python scripts/run_cdan_imu_compare.py --methods raw kabsch kabsch2 --multi_seed --epochs 30')

## 저장된 모델로 inference — R 별 target 정확도 · DTW 비교

`rotation_matrix_direct.ipynb` 와 동일한 하네스(`MMEvalCache` + `acc_dtw`). source-only(MM no-DA) 모델에 **축 정렬한 target IMU**를 넣어, 정렬 R 이 도메인 갭을 얼마나 줄이는지 본다. kabsch2 가 기존 R 들 사이 어디에 위치하는지 확인.

In [ ]:
# ── inference 준비 (다른 노트북과 동일: MM no-DA 모델 + 전체 target + DTW 레퍼런스) ──
# ★ 무거운 셀: EMG 전처리 캐시 1회 빌드. 이후 R 마다 IMU 변환만 반복.
import eval_utils as U
importlib_ok = True
N = 200  # DTW 리샘플 길이 (rotation_matrix_direct 와 동일)

model   = U.load_mm_model(os.path.join(ROOT, 'weights', 'mm_no_da_seed42_best_model.pth'), device=DEV)
dtw_ref = U.build_dtw_reference(s1, session_col='filename', time_col='timestamp')   # source 레퍼런스
df_tgt  = pd.read_parquet(os.path.join(ROOT, 'data', 'samsung2.parquet'))           # target 전체(EMG 포함)
cache   = U.MMEvalCache(df_tgt, model=model, device=DEV)
print('로드 완료 |', model.__class__.__name__, '| device', DEV, '| 캐시 윈도우', cache.n_windows)

In [ ]:
# ── R 별 target 정확도(모델 추론) + DTW(라벨-free) — 저장된 모든 R 비교 ──
import glob

def target_raw_segments(df, ref_keys, min_len=20):
    segs = {}
    for ex in ref_keys:
        sub = df[df['exercise'] == ex]
        if not len(sub):
            continue
        sess = sorted(sub['csv_filename_l'].unique())[0]   # compute_dtw 와 동일(결정적)
        mat = (sub[sub['csv_filename_l'] == sess].sort_values('Index_Time')[IMU_COLS]
               .to_numpy(np.float64))
        if len(mat) >= min_len:
            segs[ex] = mat
    return segs

tgt_raw = target_raw_segments(df_tgt, dtw_ref.keys())

def acc_dtw(R):
    """R(target→source) 의 (target 정확도%, 평균 DTW). aligned = imu @ R.T."""
    acc = cache.accuracy(lambda a, R=R: a @ R.T)
    per = [np.mean([U._dtw_mv(s, U._resample(U._zscore(tgt_raw[ex] @ R.T), N)) for s in src])
           for ex, src in dtw_ref.items() if ex in tgt_raw and src]
    return acc, float(np.mean(per))

# 저장된 모든 R_*.npy + identity 기준선
rows = [('raw (identity)', np.eye(3))]
for p in sorted(glob.glob(os.path.join(ROOT, 'results', 'R_matrices', 'R_*.npy'))):
    name = os.path.splitext(os.path.basename(p))[0].replace('R_', '')
    if name == 'raw':
        continue  # identity 와 중복
    rows.append((name, np.load(p)))

res_tbl = []
for name, Rm in rows:
    a, d = acc_dtw(Rm)
    res_tbl.append({'method': name, 'target_acc(%)': round(a, 2), 'DTW': round(d, 1)})
    print(f'  {name:16s}  acc={a:6.2f}%   dtw={d:.1f}')

res_df = (pd.DataFrame(res_tbl).sort_values('target_acc(%)', ascending=False)
          .reset_index(drop=True))
display(res_df)

# 막대그래프 (kabsch2 강조)
fig, ax = plt.subplots(figsize=(8, 4.5))
order = res_df['method'].tolist()
colors = ['tab:red' if m == 'kabsch2' else ('tab:gray' if 'raw' in m else 'tab:blue') for m in order]
bars = ax.bar(order, res_df['target_acc(%)'], color=colors, alpha=.85)
for b, v in zip(bars, res_df['target_acc(%)']):
    ax.text(b.get_x()+b.get_width()/2, v+0.3, f'{v:.1f}', ha='center', fontsize=9)
ax.set_ylabel('target accuracy (%)'); ax.set_title('축 정렬 R 별 target 정확도 (빨강=kabsch2)')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()